In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Labs"
)

xpt_file = DATA_DIR / "TRIGLY_H.xpt"
csv_file = DATA_DIR / "TRIGLY_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 TRIGLY_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXTR",
    "LBDTRSI",
    "LBDLDL",
    "LBDLDLSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "TRIGLY_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 3329:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 6:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 3,329 rows x 6 columns."
)


# ============================================================
# 4. DEFINE DECIMAL VS WHOLE-NUMBER FIELDS
# ============================================================

#
# Genuine decimal fields:
#
# WTSAF2YR
#     Fasting subsample 2-year MEC sample weight.
#
# LBDTRSI
#     Triglyceride in mmol/L.
#
# LBDLDLSI
#     LDL cholesterol in mmol/L.
#
#
# Whole-number fields:
#
# SEQN
# LBXTR
# LBDLDL
#
#
# CRITICAL:
#
# DO NOT:
#
#     round genuine decimals
#     convert genuine decimals to Int64
#     globally strip ".0"
#     use float_format in to_csv()
#

DECIMAL_COLUMNS = [
    "WTSAF2YR",
    "LBDTRSI",
    "LBDLDLSI"
]


INTEGER_COLUMNS = [
    "SEQN",
    "LBXTR",
    "LBDLDL"
]


if len(DECIMAL_COLUMNS) != 3:

    raise ValueError(
        "Expected exactly 3 genuine decimal fields."
    )


if len(INTEGER_COLUMNS) != 3:

    raise ValueError(
        "Expected exactly 3 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER globally replace it without first validating
# the exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 6 TRIGLY_H variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached TRIGLY_H.xpt:
#
# WTSAF2YR = 402
#
# TOTAL = 402
#
# CDC:
#
# WTSAF2YR = 0
#     No Lab Result
#     Count = 402
#

EXPECTED_TINY_COUNTS = {
    "WTSAF2YR": 402
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated TRIGLY_H attachment.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 402:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before correction."
    )


print(
    "\nPASS: Exact WTSAF2YR 402-value "
    "tiny-zero pattern validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED WTSAF2YR ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

wtsa_tiny_mask = (
    df["WTSAF2YR"]
    .eq(TINY_VALUE)
)


correction_count = int(
    wtsa_tiny_mask.sum()
)


if correction_count != 402:

    raise ValueError(
        "Unexpected WTSAF2YR tiny-value count."
    )


df.loc[
    wtsa_tiny_mask,
    "WTSAF2YR"
] = 0


print("\n--- VALIDATED ZERO RESTORATION ---")


print(
    "WTSAF2YR restored to zero:",
    correction_count
)


if int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
) != 402:

    raise ValueError(
        "WTSAF2YR zero restoration failed."
    )


print(
    "PASS: 402 validated WTSAF2YR zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# numeric source should contain.
#
# ONLY:
#
#     WTSAF2YR tiny -> 0
#
# is permitted to change.
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WTSAF2YR"]
    .eq(TINY_VALUE),
    "WTSAF2YR"
] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining after correction:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# MUST occur before any Int64 conversion.
#
# np.round() below is used ONLY for testing whether
# observations are fractional.
#
# It DOES NOT modify the data.
#

print("\n--- GENUINE DECIMAL DETECTION ---")


detected_decimal_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {detected_decimal_columns}"
    )


print(
    "\nPASS: Exactly 3 genuine decimal "
    "variables detected."
)


# ============================================================
# 11. DECIMAL FIELD DETAILS
# ============================================================

print("\n--- DECIMAL FIELD DETAILS ---")


for col in DECIMAL_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    print(
        f"\n{col}:"
    )


    print(
        "  Nonmissing:",
        f"{len(values):,}"
    )


    print(
        "  Fractional:",
        f"{int(fractional_mask.sum()):,}"
    )


    print(
        "  Whole-looking:",
        f"{int((~fractional_mask).sum()):,}"
    )


    print(
        "  Minimum:",
        np.min(values)
    )


    print(
        "  Maximum:",
        np.max(values)
    )


# ============================================================
# 12. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73559:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83727:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73559-83727 validated."
)


# ============================================================
# 13. CONVERT ONLY PROVEN WHOLE-NUMBER LAB FIELDS
# ============================================================

#
# Convert only:
#
#     SEQN
#     LBXTR
#     LBDLDL
#
# DO NOT convert:
#
#     WTSAF2YR
#     LBDTRSI
#     LBDLDLSI
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN, LBXTR and LBDLDL "
    "converted to nullable Int64."
)


print(
    "PASS: WTSAF2YR, LBDTRSI and LBDLDLSI "
    "remain genuine decimal fields."
)


# ============================================================
# 14. CDC CHECK - WTSAF2YR
# ============================================================

#
# Fasting Subsample 2-Year MEC Weight.
#
# CDC:
#
# Positive weight = 2,927
# Zero            =   402
# Missing         =     0
#
# 0 = No Lab Result
#
# This is a REAL decimal sample-weight variable.
#

s = df["WTSAF2YR"]


wtsa_positive = int(
    s.gt(0)
    .sum()
)


wtsa_zero = int(
    s.eq(0)
    .sum()
)


wtsa_missing = int(
    s.isna()
    .sum()
)


print("\n--- WTSAF2YR ---")


print(
    "Positive:",
    f"{wtsa_positive:,}"
)


print(
    "Zero:",
    f"{wtsa_zero:,}"
)


print(
    "Missing:",
    f"{wtsa_missing:,}"
)


print(
    "Positive minimum:",
    s[
        s > 0
    ].min()
)


print(
    "Maximum:",
    s.max()
)


if not (
    wtsa_positive == 2927
    and
    wtsa_zero == 402
    and
    wtsa_missing == 0
):

    raise ValueError(
        "WTSAF2YR differs from CDC release counts."
    )


print(
    "PASS: WTSAF2YR = "
    "2,927 positive weights + 402 valid zeros."
)


# ============================================================
# 15. CDC CHECK - LBXTR
# ============================================================

#
# Triglycerides in mg/dL.
#
# CDC:
#
# Range      = 13-4233
# Nonmissing = 3,146
# Missing    = 183
#

s = df["LBXTR"]


lbxtr_nonmissing = int(
    s.notna()
    .sum()
)


lbxtr_missing = int(
    s.isna()
    .sum()
)


lbxtr_min = int(
    s.dropna()
    .min()
)


lbxtr_max = int(
    s.dropna()
    .max()
)


print("\n--- LBXTR ---")


print(
    "Nonmissing:",
    f"{lbxtr_nonmissing:,}"
)


print(
    "Missing:",
    f"{lbxtr_missing:,}"
)


print(
    "Minimum:",
    lbxtr_min
)


print(
    "Maximum:",
    lbxtr_max
)


if not (
    lbxtr_nonmissing == 3146
    and
    lbxtr_missing == 183
    and
    lbxtr_min == 13
    and
    lbxtr_max == 4233
):

    raise ValueError(
        "LBXTR differs from CDC release."
    )


print(
    "PASS: LBXTR matches CDC."
)


# ============================================================
# 16. CDC CHECK - LBDTRSI
# ============================================================

#
# Triglycerides in mmol/L.
#
# Genuine decimal field.
#
# CDC:
#
# Range      = 0.147-47.791
# Nonmissing = 3,146
# Missing    = 183
#

s = df["LBDTRSI"]


lbdtrsi_nonmissing = int(
    s.notna()
    .sum()
)


lbdtrsi_missing = int(
    s.isna()
    .sum()
)


lbdtrsi_min = float(
    s.dropna()
    .min()
)


lbdtrsi_max = float(
    s.dropna()
    .max()
)


print("\n--- LBDTRSI ---")


print(
    "Nonmissing:",
    f"{lbdtrsi_nonmissing:,}"
)


print(
    "Missing:",
    f"{lbdtrsi_missing:,}"
)


print(
    "Minimum:",
    lbdtrsi_min
)


print(
    "Maximum:",
    lbdtrsi_max
)


if lbdtrsi_nonmissing != 3146:

    raise ValueError(
        "Unexpected LBDTRSI nonmissing count."
    )


if lbdtrsi_missing != 183:

    raise ValueError(
        "Unexpected LBDTRSI missing count."
    )


if not np.isclose(
    lbdtrsi_min,
    0.147,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDTRSI minimum."
    )


if not np.isclose(
    lbdtrsi_max,
    47.791,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDTRSI maximum."
    )


print(
    "PASS: LBDTRSI matches CDC."
)


# ============================================================
# 17. LBXTR / LBDTRSI MISSING-PATTERN CHECK
# ============================================================

trig_mg_nonmissing = (
    df["LBXTR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


trig_si_nonmissing = (
    df["LBDTRSI"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


trig_missing_mismatch = int(
    np.sum(
        trig_mg_nonmissing
        !=
        trig_si_nonmissing
    )
)


print(
    "\n--- TRIGLYCERIDE MISSING-PATTERN CHECK ---"
)


print(
    "Mismatches:",
    trig_missing_mismatch
)


if trig_missing_mismatch != 0:

    raise ValueError(
        "LBXTR and LBDTRSI missing patterns differ."
    )


print(
    "PASS: LBXTR and LBDTRSI missing "
    "patterns match exactly."
)


# ============================================================
# 18. TRIGLYCERIDE SI-CONVERSION CHECK
# ============================================================

#
# CDC:
#
#     LBDTRSI = LBXTR * 0.01129
#
# Released SI values contain three decimal places.
#
# Decimal + ROUND_HALF_UP is used ONLY for
# validation.
#
# We DO NOT overwrite LBDTRSI.
#

paired_trig = (
    df[
        [
            "LBXTR",
            "LBDTRSI"
        ]
    ]
    .dropna()
)


def triglyceride_si_expected(mg_dl):

    return float(
        (
            Decimal(
                str(
                    int(mg_dl)
                )
            )
            *
            Decimal("0.01129")
        )
        .quantize(
            Decimal("0.001"),
            rounding=ROUND_HALF_UP
        )
    )


calculated_trig_si = np.array(
    [
        triglyceride_si_expected(value)
        for value in paired_trig["LBXTR"]
    ],
    dtype=float
)


released_trig_si = (
    paired_trig["LBDTRSI"]
    .to_numpy(
        dtype=float
    )
)


trig_si_mismatch_count = int(
    np.sum(
        calculated_trig_si
        !=
        released_trig_si
    )
)


print(
    "\n--- TRIGLYCERIDE SI CONVERSION CHECK ---"
)


print(
    "Records checked:",
    f"{len(paired_trig):,}"
)


print(
    "Conversion mismatches:",
    trig_si_mismatch_count
)


if trig_si_mismatch_count != 0:

    raise ValueError(
        "LBDTRSI does not match "
        "LBXTR x 0.01129."
    )


print(
    "PASS: All 3,146 LBDTRSI values match "
    "the released triglyceride conversion."
)


# ============================================================
# 19. CDC CHECK - LBDLDL
# ============================================================

#
# LDL cholesterol in mg/dL.
#
# CDC:
#
# Range      = 14-375
# Nonmissing = 3,105
# Missing    = 224
#
# LDL is derived only when triglycerides <=400 mg/dL.
#

s = df["LBDLDL"]


ldl_nonmissing = int(
    s.notna()
    .sum()
)


ldl_missing = int(
    s.isna()
    .sum()
)


ldl_min = int(
    s.dropna()
    .min()
)


ldl_max = int(
    s.dropna()
    .max()
)


print("\n--- LBDLDL ---")


print(
    "Nonmissing:",
    f"{ldl_nonmissing:,}"
)


print(
    "Missing:",
    f"{ldl_missing:,}"
)


print(
    "Minimum:",
    ldl_min
)


print(
    "Maximum:",
    ldl_max
)


if not (
    ldl_nonmissing == 3105
    and
    ldl_missing == 224
    and
    ldl_min == 14
    and
    ldl_max == 375
):

    raise ValueError(
        "LBDLDL differs from CDC release."
    )


print(
    "PASS: LBDLDL matches CDC."
)


# ============================================================
# 20. CDC CHECK - LBDLDLSI
# ============================================================

#
# LDL cholesterol in mmol/L.
#
# Genuine decimal field.
#
# CDC:
#
# Range      = 0.362-9.698
# Nonmissing = 3,105
# Missing    = 224
#

s = df["LBDLDLSI"]


ldlsi_nonmissing = int(
    s.notna()
    .sum()
)


ldlsi_missing = int(
    s.isna()
    .sum()
)


ldlsi_min = float(
    s.dropna()
    .min()
)


ldlsi_max = float(
    s.dropna()
    .max()
)


print("\n--- LBDLDLSI ---")


print(
    "Nonmissing:",
    f"{ldlsi_nonmissing:,}"
)


print(
    "Missing:",
    f"{ldlsi_missing:,}"
)


print(
    "Minimum:",
    ldlsi_min
)


print(
    "Maximum:",
    ldlsi_max
)


if ldlsi_nonmissing != 3105:

    raise ValueError(
        "Unexpected LBDLDLSI nonmissing count."
    )


if ldlsi_missing != 224:

    raise ValueError(
        "Unexpected LBDLDLSI missing count."
    )


if not np.isclose(
    ldlsi_min,
    0.362,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDLDLSI minimum."
    )


if not np.isclose(
    ldlsi_max,
    9.698,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDLDLSI maximum."
    )


print(
    "PASS: LBDLDLSI matches CDC."
)


# ============================================================
# 21. LDL MG/DL / SI MISSING-PATTERN CHECK
# ============================================================

ldl_mg_nonmissing = (
    df["LBDLDL"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


ldl_si_nonmissing = (
    df["LBDLDLSI"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


ldl_missing_pattern_mismatch = int(
    np.sum(
        ldl_mg_nonmissing
        !=
        ldl_si_nonmissing
    )
)


if ldl_missing_pattern_mismatch != 0:

    raise ValueError(
        "LBDLDL and LBDLDLSI "
        "missing patterns differ."
    )


print(
    "PASS: LBDLDL and LBDLDLSI "
    "missing patterns match exactly."
)


# ============================================================
# 22. TRIGLYCERIDE >400 / LDL MISSING CHECK
# ============================================================

#
# CDC:
#
# Friedewald LDL calculation is valid only when:
#
#     LBXTR <= 400 mg/dL
#
# Attached file:
#
# LBXTR missing = 183
# LBXTR >400    = 41
#
# 183 + 41 = 224 missing LDL results.
#

trig_above_400 = int(
    df["LBXTR"]
    .gt(400)
    .sum()
)


trig_missing = int(
    df["LBXTR"]
    .isna()
    .sum()
)


print(
    "\n--- LDL DERIVATION ELIGIBILITY ---"
)


print(
    "LBXTR missing:",
    trig_missing
)


print(
    "LBXTR >400:",
    trig_above_400
)


print(
    "LBDLDL missing:",
    ldl_missing
)


if trig_above_400 != 41:

    raise ValueError(
        "Unexpected number of LBXTR >400 records."
    )


expected_ldl_missing_mask = (
    df["LBXTR"]
    .isna()
    |
    df["LBXTR"]
    .gt(400)
)


actual_ldl_missing_mask = (
    df["LBDLDL"]
    .isna()
)


ldl_eligibility_mismatch = int(
    np.sum(
        expected_ldl_missing_mask.to_numpy(
            dtype=bool
        )
        !=
        actual_ldl_missing_mask.to_numpy(
            dtype=bool
        )
    )
)


print(
    "LDL eligibility/missing mismatches:",
    ldl_eligibility_mismatch
)


if ldl_eligibility_mismatch != 0:

    raise ValueError(
        "LBDLDL missing positions do not match "
        "LBXTR missing or LBXTR >400."
    )


print(
    "PASS: All 224 missing LDL values are "
    "fully explained by 183 missing triglycerides "
    "+ 41 triglycerides >400."
)


# ============================================================
# 23. LDL SI-CONVERSION CHECK
# ============================================================

#
# CDC:
#
#     LBDLDLSI = LBDLDL * 0.02586
#
# Validation only.
#
# DO NOT overwrite released LBDLDLSI.
#

paired_ldl = (
    df[
        [
            "LBDLDL",
            "LBDLDLSI"
        ]
    ]
    .dropna()
)


def ldl_si_expected(mg_dl):

    return float(
        (
            Decimal(
                str(
                    int(mg_dl)
                )
            )
            *
            Decimal("0.02586")
        )
        .quantize(
            Decimal("0.001"),
            rounding=ROUND_HALF_UP
        )
    )


calculated_ldl_si = np.array(
    [
        ldl_si_expected(value)
        for value in paired_ldl["LBDLDL"]
    ],
    dtype=float
)


released_ldl_si = (
    paired_ldl["LBDLDLSI"]
    .to_numpy(
        dtype=float
    )
)


ldl_si_mismatch_count = int(
    np.sum(
        calculated_ldl_si
        !=
        released_ldl_si
    )
)


print(
    "\n--- LDL SI CONVERSION CHECK ---"
)


print(
    "Records checked:",
    f"{len(paired_ldl):,}"
)


print(
    "Conversion mismatches:",
    ldl_si_mismatch_count
)


if ldl_si_mismatch_count != 0:

    raise ValueError(
        "LBDLDLSI does not match "
        "LBDLDL x 0.02586."
    )


print(
    "PASS: All 3,105 LDL SI values "
    "match the released conversion."
)


# ============================================================
# 24. FINAL ZERO PATTERN
# ============================================================

#
# Original pandas-decoded file had no ordinary zeros.
#
# Therefore the ONLY zeros after correction should be:
#
# WTSAF2YR = 402
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated TRIGLY_H pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {final_zero_counts}"
    )


print(
    "PASS: Final zero pattern contains "
    "exactly 402 WTSAF2YR zeros."
)


# ============================================================
# 25. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY allowed numeric modification:
#
#     WTSAF2YR:
#     402 tiny values -> 0
#
# All other numeric values must remain identical.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 6 variables match the exact "
    "expected corrected XPT values."
)


# ============================================================
# 26. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
# NO:
#
#     manual decimal-string formatting
#
# This preserves:
#
# WTSAF2YR
# LBDTRSI
# LBDLDLSI
#
# as genuine decimal fields.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)


print(
    csv_file
)


# ============================================================
# 27. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 28. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 6 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 29. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 30. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 31. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 32. .0 CHECK - WHOLE-NUMBER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT globally search this file for ".0".
#
# Genuine decimal fields:
#
#     WTSAF2YR
#     LBDTRSI
#     LBDLDLSI
#
# These may legitimately contain ".0".
#
# Check ONLY:
#
#     SEQN
#     LBXTR
#     LBDLDL
#

print(
    "\n--- INTEGER FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "SEQN, LBXTR or LBDLDL."
)


# ============================================================
# 33. LEGITIMATE DECIMAL .0 VALUES
# ============================================================

#
# INFORMATIONAL ONLY.
#
# ".0" is allowed in the three genuine decimal fields.
#
# DO NOT treat it as an error.
#

print(
    "\n--- LEGITIMATE DECIMAL .0 TOKENS ---"
)


decimal_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col]
            .ne("")
            &
            raw_csv_df[col]
            .str.endswith(".0")
        )
        .sum()
    )


    decimal_dot_zero_counts[col] = count


    print(
        f"{col}: {count:,}"
    )


total_decimal_dot_zero = sum(
    decimal_dot_zero_counts.values()
)


print(
    "\nTotal legitimate '.0' tokens "
    "inside decimal fields:",
    f"{total_decimal_dot_zero:,}"
)


print(
    "PASS: Genuine Lab decimal fields "
    "were not globally reformatted."
)


# ============================================================
# 34. EXACT DECIMAL FIDELITY
# ============================================================

print(
    "\n--- EXACT DECIMAL FIDELITY ---"
)


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Decimal fidelity failed:\n"
        + str(decimal_failures)
    )


print(
    "PASS: WTSAF2YR, LBDTRSI and LBDLDLSI "
    "preserved exactly."
)


# ============================================================
# 35. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "402 WTSAF2YR zeros."
)


# ============================================================
# 36. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 37. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73559:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83727:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 38. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "TRIGLY_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Laboratory - LDL & Triglycerides"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "WTSAF2YR zero restorations:",
    correction_count
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number variables:",
    INTEGER_COLUMNS
)


print(
    "LBXTR nonmissing:",
    f"{lbxtr_nonmissing:,}"
)


print(
    "LBXTR range:",
    f"{lbxtr_min}-{lbxtr_max} mg/dL"
)


print(
    "LBDTRSI range:",
    f"{lbdtrsi_min}-{lbdtrsi_max} mmol/L"
)


print(
    "LBDLDL nonmissing:",
    f"{ldl_nonmissing:,}"
)


print(
    "LBDLDL range:",
    f"{ldl_min}-{ldl_max} mg/dL"
)


print(
    "LBDLDLSI range:",
    f"{ldlsi_min}-{ldlsi_max} mmol/L"
)


print(
    "Triglycerides >400:",
    trig_above_400
)


print(
    "Triglyceride SI conversion mismatches:",
    trig_si_mismatch_count
)


print(
    "LDL SI conversion mismatches:",
    ldl_si_mismatch_count
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Decimal differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0' in integer fields: "
    "0 expected"
)


print(
    "Legitimate '.0' in decimal fields: "
    "ALLOWED"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 TRIGLY_H XPT -> CSV
Rows: 3,329
Columns: 6
PASS: Structure = 3,329 rows x 6 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['WTSAF2YR', 'LBDTRSI', 'LBDLDLSI']
Whole-number fields: ['SEQN', 'LBXTR', 'LBDLDL']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 402

Tiny values by variable:
WTSAF2YR    402

PASS: Exact WTSAF2YR 402-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
WTSAF2YR restored to zero: 402
PASS: 402 validated WTSAF2YR zeros restored.

Tiny values remaining after correction: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
WTSAF2YR: 2,927 fractional observations
LBXTR: 0 fractional observations
LBDTRSI: 3,146 fractional observations
LBDLDL: 0 fractional observations
LBDLDLSI: 3,067 fractional observations

PASS: Exactly 3 genuine decimal variables detected.

--- DECIMAL FIELD DETAILS ---

WTSAF2YR:
  Nonmissing: 3,3